# Tiki Fashion Marketplace Pulse - Week 1

## Exploratory Data Analysis

### Project Question

What characteristics commonly appear among high-selling fashion products on Tiki, and how do these patterns differ across product groups?

### Week 1 Workflow

1. Understand the datasets
2. Inspect data structure
3. Check data quality
4. Prepare and combine data
5. Explore product performance
6. Create visualizations
7. Write preliminary findings

In [64]:
# Import libraries for data analysis and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [65]:
# Define project paths for raw data, processed data, and figures
RAW_DATA_PATH = Path("../data/raw")
PROCESSED_DATA_PATH = Path("../data/processed")
FIGURES_PATH = Path("../figures")

## 1. Dataset Understanding

Load and inspect the six raw Tiki fashion datasets before any cleaning or transformation.

In [66]:
# Find the raw CSV files and verify the number of available datasets
csv_files = sorted(RAW_DATA_PATH.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

Number of CSV files: 6


In [67]:
# Load each raw CSV file into a dictionary of DataFrames
datasets = {}

for file in csv_files:
    datasets[file.stem] = pd.read_csv(file)

### 1.1 Dataset Overview

In [68]:
# Summarize the number of rows and columns in each dataset
dataset_summary = []

for name, df in datasets.items():
    dataset_summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

dataset_summary = pd.DataFrame(dataset_summary)

dataset_summary

,dataset,rows,columns
0,vietnamese_tiki_products_backpacks_suitcases,5361,19
1,vietnamese_tiki_products_fashion_accessories,16019,19
2,vietnamese_tiki_products_men_bags,4234,19
3,vietnamese_tiki_products_men_shoes,5745,19
4,vietnamese_tiki_products_women_bags,4325,19
5,vietnamese_tiki_products_women_shoes,5919,19


In [69]:
# Calculate the total number of records across all six datasets
print("Total rows:", dataset_summary["rows"].sum())

Total rows: 41603


### 1.2 Schema Consistency

In [70]:
# Check whether all datasets have the same column structure
reference_columns = list(next(iter(datasets.values())).columns)

same_schema = all(
    list(df.columns) == reference_columns
    for df in datasets.values()
)

print("All datasets have the same columns:", same_schema)

All datasets have the same columns: True


In [71]:
reference_columns

['Unnamed: 0',
 'id',
 'name',
 'description',
 'original_price',
 'price',
 'fulfillment_type',
 'brand',
 'review_count',
 'rating_average',
 'favourite_count',
 'pay_later',
 'current_seller',
 'date_created',
 'number_of_images',
 'vnd_cashback',
 'has_video',
 'category',
 'quantity_sold']

In [72]:
# Check whether corresponding columns use the same data types
reference_dtypes = next(iter(datasets.values())).dtypes

same_dtypes = all(
    df.dtypes.equals(reference_dtypes)
    for df in datasets.values()
)

print("All datasets have the same data types:", same_dtypes)

All datasets have the same data types: True


In [73]:
reference_dtypes

Unnamed: 0            int64
id                    int64
name                    str
description             str
original_price        int64
price                 int64
fulfillment_type        str
brand                   str
review_count          int64
rating_average      float64
favourite_count       int64
pay_later              bool
current_seller          str
date_created          int64
number_of_images      int64
vnd_cashback          int64
has_video              bool
category                str
quantity_sold         int64
dtype: object

### 1.3 Product ID Check

In [74]:
# Count unique and repeated product IDs in each dataset
id_summary = []

for name, df in datasets.items():
    id_summary.append({
        "dataset": name,
        "rows": len(df),
        "unique_ids": df["id"].nunique(),
        "duplicate_ids": df["id"].duplicated().sum()
    })

id_summary = pd.DataFrame(id_summary)

id_summary

,dataset,rows,unique_ids,duplicate_ids
0,vietnamese_tiki_products_backpacks_suitcases,5361,5359,2
1,vietnamese_tiki_products_fashion_accessories,16019,16005,14
2,vietnamese_tiki_products_men_bags,4234,4227,7
3,vietnamese_tiki_products_men_shoes,5745,5744,1
4,vietnamese_tiki_products_women_bags,4325,4324,1
5,vietnamese_tiki_products_women_shoes,5919,5917,2


### Initial Structure Findings

- The project contains six Tiki fashion datasets.
- The datasets differ in the number of product records.
- All six datasets contain the same 19 columns.
- Their consistent structure makes them suitable for later combination.
- Product `id` will be examined as the main candidate for identifying individual products.
- When the datasets are combined, a `source_group` variable should be added to preserve the original product group.

### 1.4 Duplicate Product ID Inspection

Some datasets contain duplicated product IDs.  
Before deciding how to handle them, the duplicated records are inspected to determine whether they represent exact duplicate rows or different records sharing the same product ID.

In [75]:
# Display all rows whose product ID appears more than once
for name, df in datasets.items():
    duplicate_rows = df[df["id"].duplicated(keep=False)]

    if not duplicate_rows.empty:
        print(f"\n{name}")
        display(
            duplicate_rows.sort_values("id")
        )


vietnamese_tiki_products_backpacks_suitcases


,Unnamed: 0,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold
4962,4962,169624384,vali màu hồng nhạt 24 inch,vali kéo du lịch cao cấp thời trang S...,380000,380000,dropship,OEM,2,4.5,0,True,Bagsell Fashion,546,1,0,False,Balo và Vali,4
5234,5234,169624384,vali màu hồng nhạt 24 inch,vali kéo du lịch cao cấp thời trang S...,380000,380000,dropship,OEM,2,4.5,0,True,Bagsell Fashion,546,1,0,False,Balo và Vali,4
4562,4562,182797637,Áo Trùm Vải Không Dệt Vali SAKOS,- Sản phẩm được thiết kế và sản xuất theo tiêu...,239000,155000,dropship,Sakos,5,4.8,0,True,SAKOS Official Store,474,3,0,False,Balo và Vali,33
5268,5268,182797637,Áo Trùm Vải Không Dệt Vali SAKOS,- Sản phẩm được thiết kế và sản xuất theo tiêu...,239000,155000,dropship,Sakos,5,4.8,0,True,SAKOS Official Store,474,3,0,False,Balo và Vali,33



vietnamese_tiki_products_fashion_accessories


,Unnamed: 0,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold
2229,2229,138283799,"Băng đô rửa mặt, Bờm cài tóc đính Nơ phối ren ...","***Mẫu: Băng đô rửa mặt, Bờm cài tóc đí...",60000,42000,dropship,OEM,1,5.0,0,False,CULI ACCESSORIES,672,14,0,True,Phụ kiện thời trang nữ,1
8464,8464,138283799,"Băng đô rửa mặt, Bờm cài tóc đính Nơ phối ren ...","***Mẫu: Băng đô rửa mặt, Bờm cài tóc đí...",60000,42000,dropship,OEM,1,5.0,0,False,CULI ACCESSORIES,672,14,0,True,Phụ kiện thời trang nữ,1
6624,6624,160177797,"Ghim chữ U cài tóc, búi tóc 7cm bọc sơn đen c...","Ghim chữ U cài tóc, búi tóc 7cm bọc sơn đen ch...",12000,12000,dropship,OEM,0,0.0,0,False,An Nhiên Beauty And Fashion,599,6,325,False,Phụ kiện thời trang nữ,8
526,526,160177797,"Ghim chữ U cài tóc, búi tóc 7cm bọc sơn đen c...","Ghim chữ U cài tóc, búi tóc 7cm bọc sơn đen ch...",12000,12000,dropship,OEM,0,0.0,0,False,An Nhiên Beauty And Fashion,599,6,328,False,Phụ kiện thời trang nữ,8
4923,4923,162026679,Mắt kính nobita phong cách retro siêu dễ thươn...,Chất Liệu gọng: Kim Loại + NhựaChất Liệu ống k...,35000,35000,dropship,OEM,4,4.0,0,False,ANCHI HOUSE,576,8,937,True,Mắt kính,8
2836,2836,162026679,Mắt kính nobita phong cách retro siêu dễ thươn...,Chất Liệu gọng: Kim Loại + NhựaChất Liệu ống k...,35000,35000,dropship,OEM,4,4.0,0,False,ANCHI HOUSE,576,8,943,True,Mắt kính,8
8401,8401,167964878,Khăn bandana nam in hoa văn vải cotton size 55...,KHĂN BANDANA KHĂN TURBAN NAM Giá 60k 1 chi...,60000,60000,dropship,OEM,1,5.0,0,False,Maben Khăn phụ kiện và Quà tặng,494,9,0,False,Phụ kiện thời trang,1
4182,4182,167964878,Khăn bandana nam in hoa văn vải cotton size 55...,KHĂN BANDANA KHĂN TURBAN NAM Giá 60k 1 chi...,60000,60000,dropship,OEM,1,5.0,0,False,Maben Khăn phụ kiện và Quà tặng,494,9,0,False,Phụ kiện thời trang,1
7490,7490,168528667,Tất trơn sọc ngang 2 vạch điểm nhấn unisex cổ ...,GIỚI THIỆU SẢN PHẨM TẤT VỚ Tất - Vớ g...,11000,11000,dropship,OEM,0,0.0,0,False,Aomi Store,551,6,0,False,Phụ kiện thời trang nữ,0
8060,8060,168528667,Tất trơn sọc ngang 2 vạch điểm nhấn unisex cổ ...,GIỚI THIỆU SẢN PHẨM TẤT VỚ Tất - Vớ g...,11000,11000,dropship,OEM,0,0.0,0,False,Aomi Store,551,6,0,False,Phụ kiện thời trang nữ,0



vietnamese_tiki_products_men_bags


...

,Unnamed: 0,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold
2693,2693,179122186,Dép Nam PEAK TAICHI VOID SANDALS ET22857L,Thông tin sản phẩm:Thương hiệu: PeakChức năng:...,990000,990000,dropship,Peak,1,5.0,0,True,Peak Official Store,489,11,0,False,Giày - Dép nam,4
3652,3652,179122186,Dép Nam PEAK TAICHI VOID SANDALS ET22857L,Thông tin sản phẩm:Thương hiệu: PeakChức năng:...,990000,990000,dropship,Peak,1,5.0,0,True,Peak Official Store,489,11,0,False,Giày - Dép nam,4



vietnamese_tiki_products_women_bags


,Unnamed: 0,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold
267,267,148967272,Túi cói Merci sang chảnh,Túi Cói Merci Nhỏ nhỏ xinh xinh ...,44980,42731,dropship,OEM,0,0.0,0,False,Hitech07,637,3,1160,False,Túi thời trang nữ,2
3367,3367,148967272,Túi cói Merci sang chảnh,Túi Cói Merci Nhỏ nhỏ xinh xinh ...,44980,42731,dropship,OEM,0,0.0,0,False,Hitech07,637,3,1160,False,Túi thời trang nữ,2



vietnamese_tiki_products_women_shoes


,Unnamed: 0,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold
2459,2459,181997031,Sandal nữ nhựa siêu bền mũi nhọn đính đá phong...,Sandal nữ nhựa siêu bền mũi nhọn đính đá phong...,88000,88000,dropship,OEM,0,0.0,0,False,Phatdat,478,7,0,False,Giày - Dép nữ,1
3095,3095,181997031,Sandal nữ nhựa siêu bền mũi nhọn đính đá phong...,Sandal nữ nhựa siêu bền mũi nhọn đính đá phong...,88000,88000,dropship,OEM,0,0.0,0,False,Phatdat,478,7,0,False,Giày - Dép nữ,1
1552,1552,192490798,Dép kẹp đế xuồng nhung 7f,Dép kẹp đế xuồng nhung 7fsize 35 đến 39 ...,109000,109000,dropship,GPSHOP,1,1.0,0,True,THẢO NGỌC SHOP,412,6,0,False,Giày - Dép nữ,10
3578,3578,192490798,Dép kẹp đế xuồng nhung 7f,Dép kẹp đế xuồng nhung 7fsize 35 đến 39 ...,109000,109000,dropship,GPSHOP,1,1.0,0,True,THẢO NGỌC SHOP,412,6,0,False,Giày - Dép nữ,10


In [76]:
# Count exact duplicate records after excluding the technical index column
exact_duplicate_summary = []

for name, df in datasets.items():
    comparison_columns = [
        column for column in df.columns
        if column != "Unnamed: 0"
    ]

    exact_duplicate_summary.append({
        "dataset": name,
        "duplicate_group_rows": df.duplicated(
            subset=comparison_columns,
            keep=False
        ).sum(),
        "redundant_rows": df.duplicated(
            subset=comparison_columns,
            keep="first"
        ).sum()
    })

exact_duplicate_summary = pd.DataFrame(exact_duplicate_summary)

exact_duplicate_summary

,dataset,duplicate_group_rows,redundant_rows
0,vietnamese_tiki_products_backpacks_suitcases,4,2
1,vietnamese_tiki_products_fashion_accessories,20,10
2,vietnamese_tiki_products_men_bags,12,6
3,vietnamese_tiki_products_men_shoes,2,1
4,vietnamese_tiki_products_women_bags,2,1
5,vietnamese_tiki_products_women_shoes,4,2


### Research Question 1 — What does one row represent?

Each row appears to represent one Tiki fashion product listing and contains information about the product, pricing, seller, reviews, presentation features, and sales.

The `id` column is the strongest candidate for identifying a product. However, duplicated IDs are present in some source files, so the identifier must be investigated further before being treated as globally unique.

### Research Question 2 — How do the six datasets differ?

The six datasets represent different fashion product groups and contain different numbers of product records. However, all six datasets share the same 19-column structure and consistent data types.

When the datasets are combined, a `source_group` variable should be added so that the original product group of each row is preserved.

### Research Question 3 — What are the main variables and their types?

In [77]:
# Create a data dictionary for the original dataset variables
data_dictionary = pd.DataFrame([
    ["Unnamed: 0", "Index / Technical", "Index column from the source file"],
    ["id", "Identifier", "Product identifier"],
    ["name", "Text", "Product name"],
    ["description", "Text", "Product description"],
    ["original_price", "Numerical", "Original listed price"],
    ["price", "Numerical", "Current selling price"],
    ["fulfillment_type", "Categorical", "Fulfillment method"],
    ["brand", "Categorical", "Product brand"],
    ["review_count", "Numerical", "Number of reviews"],
    ["rating_average", "Numerical", "Average product rating"],
    ["favourite_count", "Numerical", "Number of favourites"],
    ["pay_later", "Boolean", "Whether pay-later is available"],
    ["current_seller", "Categorical", "Current seller"],
    ["date_created", "Numerical / Unknown", "Raw numeric creation-related field; meaning and unit require further clarification"],
    ["number_of_images", "Numerical", "Number of product images"],
    ["vnd_cashback", "Numerical", "Cashback amount in VND"],
    ["has_video", "Boolean", "Whether the product has a video"],
    ["category", "Categorical", "Product category"],
    ["quantity_sold", "Numerical", "Number of units sold"]
], columns=["variable", "analytical_type", "description"])

data_dictionary

,variable,analytical_type,description
0,Unnamed: 0,Index / Technical,Index column from the source file
1,id,Identifier,Product identifier
2,name,Text,Product name
3,description,Text,Product description
4,original_price,Numerical,Original listed price
5,price,Numerical,Current selling price
6,fulfillment_type,Categorical,Fulfillment method
7,brand,Categorical,Product brand
8,review_count,Numerical,Number of reviews
9,rating_average,Numerical,Average product rating


The dataset contains numerical, categorical, boolean, text, identifier, technical, and unknown-type variables.

Numerical variables include variables such as `price`, `original_price`, `review_count`, `rating_average`, `number_of_images`, `vnd_cashback`, and `quantity_sold`.

Categorical variables include `fulfillment_type`, `brand`, `current_seller`, and `category`.

Boolean variables include `pay_later` and `has_video`, while `name` and `description` are text variables.

The `id` column serves as a product identifier candidate, while `Unnamed: 0` appears to be a technical index column and will be examined further during the data quality stage.

### Research Question 4 — What does `quantity_sold` represent?
`quantity_sold` represents the number of units sold for a product and is used as the main product performance variable in this project.

However, `quantity_sold` is not the same as revenue or profit.

Revenue depends on both the number of units sold and the selling price, while profit also depends on costs and other expenses that are not available in this dataset.

Therefore, Week 1 uses `quantity_sold` as a sales-volume indicator rather than a complete measure of financial performance.

## 2. Data Quality & Preparation

This section examines missing values, duplicate product IDs, data types, unusual values, and other potential data quality issues before any cleaning decisions are made.

### 2.1 Missing Values

In [78]:
# Summarize missing values in each dataset
missing_summary = []

for name, df in datasets.items():
    missing_summary.append({
        "dataset": name,
        "total_rows": len(df),
        "missing_cells": df.isna().sum().sum(),
        "columns_with_missing": (df.isna().sum() > 0).sum()
    })

missing_summary = pd.DataFrame(missing_summary)

missing_summary

,dataset,total_rows,missing_cells,columns_with_missing
0,vietnamese_tiki_products_backpacks_suitcases,5361,0,0
1,vietnamese_tiki_products_fashion_accessories,16019,2,1
2,vietnamese_tiki_products_men_bags,4234,0,0
3,vietnamese_tiki_products_men_shoes,5745,1,1
4,vietnamese_tiki_products_women_bags,4325,0,0
5,vietnamese_tiki_products_women_shoes,5919,0,0


In [79]:
# Display the columns containing missing values in each dataset
for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")

    if missing.empty:
        print("No missing values")
    else:
        display(missing.to_frame("missing_count"))


vietnamese_tiki_products_backpacks_suitcases
No missing values

vietnamese_tiki_products_fashion_accessories


,missing_count
brand,2



vietnamese_tiki_products_men_bags
No missing values

vietnamese_tiki_products_men_shoes


,missing_count
fulfillment_type,1



vietnamese_tiki_products_women_bags
No missing values

vietnamese_tiki_products_women_shoes
No missing values


### 2.2 Duplicate Product IDs

The earlier product ID inspection found duplicated IDs in some source datasets.

These duplicated IDs require further investigation before any records are removed, because duplicated product IDs do not necessarily mean that the entire rows are exact duplicates.

### 2.3 Data Type Consistency

The six source datasets use consistent pandas data types for corresponding columns.

However, some variables still require semantic review. The `date_created` column is stored as an integer, but its meaning and unit are unclear. It should not be converted to datetime unless its encoding can be confirmed.

`Unnamed: 0` appears to be a technical index rather than an analytical variable.

In [80]:
# Verify whether Unnamed: 0 duplicates the DataFrame row index
unnamed_summary = []

for name, df in datasets.items():
    expected_index = pd.Series(range(len(df)), index=df.index)

    unnamed_summary.append({
        "dataset": name,
        "unique_values": df["Unnamed: 0"].nunique(),
        "matches_row_index": df["Unnamed: 0"].equals(expected_index)
    })

unnamed_summary = pd.DataFrame(unnamed_summary)

unnamed_summary

,dataset,unique_values,matches_row_index
0,vietnamese_tiki_products_backpacks_suitcases,5361,True
1,vietnamese_tiki_products_fashion_accessories,16019,True
2,vietnamese_tiki_products_men_bags,4234,True
3,vietnamese_tiki_products_men_shoes,5745,True
4,vietnamese_tiki_products_women_bags,4325,True
5,vietnamese_tiki_products_women_shoes,5919,True


### Initial Data Quality Notes

- Missing values are present in some variables and will be evaluated column by column before treatment.
- Some duplicated product IDs were found and require inspection before deciding whether records should be removed.
- Data types are structurally consistent across the six files, although some columns may require semantic conversion.
- `Unnamed: 0` matches the row index in all six datasets and is therefore a redundant technical index column. It can be removed from the processed dataset while the raw files remain unchanged.

### Research Question 5 — Are there missing values, duplicated product IDs, or inappropriate data types?

Only three missing values were found: two in `brand` and one in `fulfillment_type`.

Some product IDs are duplicated. After ignoring the technical `Unnamed: 0` column, there are 44 records in duplicate groups, equal to 22 extra duplicate rows. However, some duplicated IDs have different values, so they should not be removed based on ID alone.

Data types are generally consistent across the six datasets. However, `date_created` is stored as an integer and its meaning is unclear, so it should not yet be treated as a datetime variable.

### 2.4 Price Anomaly Inspection

In [81]:
# Count zero and negative values in current and original prices
price_anomaly_summary = []

for name, df in datasets.items():
    price_anomaly_summary.append({
        "dataset": name,
        "price_zero": (df["price"] == 0).sum(),
        "price_negative": (df["price"] < 0).sum(),
        "original_price_zero": (df["original_price"] == 0).sum(),
        "original_price_negative": (df["original_price"] < 0).sum()
    })

price_anomaly_summary = pd.DataFrame(price_anomaly_summary)

price_anomaly_summary

,dataset,price_zero,price_negative,original_price_zero,original_price_negative
0,vietnamese_tiki_products_backpacks_suitcases,0,0,0,0
1,vietnamese_tiki_products_fashion_accessories,0,0,0,0
2,vietnamese_tiki_products_men_bags,0,0,0,0
3,vietnamese_tiki_products_men_shoes,0,0,0,0
4,vietnamese_tiki_products_women_bags,1,0,1,0
5,vietnamese_tiki_products_women_shoes,0,0,0,0


In [82]:
# Compare current prices with original prices
price_relationship_summary = []

for name, df in datasets.items():
    price_relationship_summary.append({
        "dataset": name,
        "price_above_original": (df["price"] > df["original_price"]).sum(),
        "price_equal_original": (df["price"] == df["original_price"]).sum(),
        "price_below_original": (df["price"] < df["original_price"]).sum()
    })

price_relationship_summary = pd.DataFrame(price_relationship_summary)

price_relationship_summary

,dataset,price_above_original,price_equal_original,price_below_original
0,vietnamese_tiki_products_backpacks_suitcases,0,3820,1541
1,vietnamese_tiki_products_fashion_accessories,0,13134,2885
2,vietnamese_tiki_products_men_bags,0,2877,1357
3,vietnamese_tiki_products_men_shoes,0,4519,1226
4,vietnamese_tiki_products_women_bags,0,3537,788
5,vietnamese_tiki_products_women_shoes,0,4632,1287


In [83]:
# Calculate price summary statistics for each dataset
price_summary = []

for name, df in datasets.items():
    price_summary.append({
        "dataset": name,
        "min_price": df["price"].min(),
        "median_price": df["price"].median(),
        "mean_price": df["price"].mean(),
        "max_price": df["price"].max()
    })

price_summary = pd.DataFrame(price_summary)

price_summary

,dataset,min_price,median_price,mean_price,max_price
0,vietnamese_tiki_products_backpacks_suitcases,1000,159000.0,497216.379407,18840000
1,vietnamese_tiki_products_fashion_accessories,1000,45000.0,82030.159810,11900000
2,vietnamese_tiki_products_men_bags,1350,185000.0,580201.698158,19800000
3,vietnamese_tiki_products_men_shoes,4000,180000.0,372459.547433,14700000
4,vietnamese_tiki_products_women_bags,0,69000.0,247528.782890,17500000
5,vietnamese_tiki_products_women_shoes,1000,158000.0,284161.841189,5500000


In [84]:
# Inspect the five highest-priced products in each dataset
for name, df in datasets.items():
    print(f"\n{name}")

    display(
        df.nlargest(5, "price")[
            ["id", "name", "original_price", "price", "quantity_sold"]
        ]
    )


vietnamese_tiki_products_backpacks_suitcases


,id,name,original_price,price,quantity_sold
4531,205147975,Vali hợp kim nhôm nguyên khối MS1317 Macsim ca...,18840000,18840000,0
4532,179944053,Vali hợp kim nhôm nguyên khối MS1317 Macsim ca...,18840000,18840000,0
4605,205147850,Vali hợp kim nhôm nguyên khối MS1317 Macsim ca...,18840000,18840000,0
4964,184253863,Vali xách tay Bric’s Bellagio 21″ – Xanh Olive...,17680000,17680000,0
5060,184252647,Vali xách tay Bric’s Bellagio 21″ – Xanh Olive...,17680000,17680000,0



vietnamese_tiki_products_fashion_accessories


,id,name,original_price,price,quantity_sold
8736,175539164,"Máy Sấy Tóc Dyson HD07, Máy Tạo Kiểu Tóc Công ...",11900000,11900000,0
13410,205147773,Mũ cao bồi miền tây nước Mỹ chất liệu da thô n...,5200000,5200000,0
13423,195756623,Mũ cao bồi miền tây nước Mỹ chất liệu da thô n...,5200000,5200000,0
13669,184737933,Mũ cao bồi - Chất liệu da thô nguyên miếng cao...,5200000,5200000,1
13888,252633512,Mũ Jacket da cao cấp NÓN SƠN chính hãng MC028B...,5000000,5000000,0



vietnamese_tiki_products_men_bags


...

,id,name,original_price,price,quantity_sold
5297,192378216,Kính hiển vi 3 mắt BS 1153 EPL Bscope Euromex ...,14700000,14700000,0
2546,166011357,Giày Palladium GAO EVA HIGH BOOT - 07739-008-M,11000000,11000000,0
2264,135472219,Original Timberland Giày Cổ Cao Nam 6-inch Pre...,6490000,6490000,0
2309,249859398,[NEW] Original TIMBERLAND Giày Cổ Cao Nam 6-in...,6490000,6490000,0
2318,249861817,[NEW] Original TIMBERLAND Giày Cổ Cao Nam 6-in...,6490000,6490000,0



vietnamese_tiki_products_women_bags


,id,name,original_price,price,quantity_sold
3672,147725085,Túi đeo chéo da cá sấu nữ VIETPHONG Aline T183,17500000,17500000,0
3501,147725162,Túi xách da cá sấu nữ VIETPHONG Large Phyllis ...,16999000,16999000,0
3578,147724874,Túi xách da cá sấu nữ VIETPHONG Millie T166,16999000,16999000,0
3673,147724884,Túi xách da cá sấu nữ VIETPHONG Emelia T193,14549000,14549000,0
3637,147724959,Túi xách da cá sấu nữ VIETPHONG Victoria Curve...,14119000,14119000,0



vietnamese_tiki_products_women_shoes


,id,name,original_price,price,quantity_sold
2223,177662952,Giày Boot Nữ GEOX D Bigliana 90 D,5500000,5500000,0
2250,249051795,Original Timberland Giày Nữ Cổ Trung Nellie Wa...,5490000,5490000,0
4832,252266600,Giày Sneakers Nữ GEOX D Kristene A,5400000,5400000,0
4896,252266092,Giày Sneakers Nữ GEOX D Kristene A,5400000,5400000,0
3194,258635926,Giày Cao Gót Nữ GEOX D Walk Pleasure 85 A,5200000,5200000,0


#### Potential Category Anomalies

A quick review found some products that may be in the wrong group, such as a microscope in Men Shoes and a hair dryer in Fashion Accessories. These products will be noted but not removed without further evidence.

In [85]:
# Inspect the Women Bags record with zero current and original price
women_bags = datasets["vietnamese_tiki_products_women_bags"]

women_bags[
    (women_bags["price"] == 0) |
    (women_bags["original_price"] == 0)
][
    [
        "id",
        "name",
        "original_price",
        "price",
        "quantity_sold",
        "brand",
        "current_seller"
    ]
]

,id,name,original_price,price,quantity_sold,brand,current_seller
3179,129373206,[Gift] Túi Canvas Morra,0,0,3,MORRA,M RRA Official Store


### Research Question 6 — Are there unusual price values or potentially unreasonable records?

Most price values appear reasonable. No negative prices or cases where `price` is greater than `original_price` were found.

One Women Bags record has both `price = 0` and `original_price = 0`. The product name indicates that it is a gift item, so the zero price may be intentional rather than an error.

Some products have very high prices, but many appear to be premium products. Therefore, extreme prices should not be removed automatically.

### 2.5 Near-Constant Variable Inspection

In [86]:
# Count distinct values to identify constant or near-constant variables
unique_value_summary = []

for name, df in datasets.items():
    for column in df.columns:
        unique_value_summary.append({
            "dataset": name,
            "column": column,
            "unique_values": df[column].nunique(dropna=False)
        })

unique_value_summary = pd.DataFrame(unique_value_summary)

unique_value_summary[
    unique_value_summary["unique_values"] <= 5
].sort_values(["column", "dataset"])

,dataset,column,unique_values
10,vietnamese_tiki_products_backpacks_suitcases,favourite_count,1
29,vietnamese_tiki_products_fashion_accessories,favourite_count,1
48,vietnamese_tiki_products_men_bags,favourite_count,1
67,vietnamese_tiki_products_men_shoes,favourite_count,1
86,vietnamese_tiki_products_women_bags,favourite_count,1
105,vietnamese_tiki_products_women_shoes,favourite_count,1
6,vietnamese_tiki_products_backpacks_suitcases,fulfillment_type,3
25,vietnamese_tiki_products_fashion_accessories,fulfillment_type,3
44,vietnamese_tiki_products_men_bags,fulfillment_type,3
63,vietnamese_tiki_products_men_shoes,fulfillment_type,4


In [87]:
# Confirm whether favourite_count is constant across all datasets
for name, df in datasets.items():
    print(
        name,
        "->",
        df["favourite_count"].unique()
    )

vietnamese_tiki_products_backpacks_suitcases -> [0]
vietnamese_tiki_products_fashion_accessories -> [0]
vietnamese_tiki_products_men_bags -> [0]
vietnamese_tiki_products_men_shoes -> [0]
vietnamese_tiki_products_women_bags -> [0]
vietnamese_tiki_products_women_shoes -> [0]


In [88]:
# Measure how strongly each variable is dominated by its most common value
dominance_summary = []

for name, df in datasets.items():
    for column in df.columns:
        value_distribution = df[column].value_counts(
            dropna=False,
            normalize=True
        )

        dominance_summary.append({
            "dataset": name,
            "column": column,
            "unique_values": df[column].nunique(dropna=False),
            "most_common_value": value_distribution.index[0],
            "dominant_percentage": value_distribution.iloc[0] * 100
        })

dominance_summary = pd.DataFrame(dominance_summary)

# Find variables dominated by one value in at least 95% of records
near_constant_candidates = dominance_summary[
    dominance_summary["dominant_percentage"] >= 95
].sort_values(
    ["dominant_percentage", "column"],
    ascending=[False, True]
)

near_constant_candidates

,dataset,column,unique_values,most_common_value,dominant_percentage
10,vietnamese_tiki_products_backpacks_suitcases,favourite_count,1,0,100.000000
29,vietnamese_tiki_products_fashion_accessories,favourite_count,1,0,100.000000
48,vietnamese_tiki_products_men_bags,favourite_count,1,0,100.000000
67,vietnamese_tiki_products_men_shoes,favourite_count,1,0,100.000000
86,vietnamese_tiki_products_women_bags,favourite_count,1,0,100.000000
105,vietnamese_tiki_products_women_shoes,favourite_count,1,0,100.000000
63,vietnamese_tiki_products_men_shoes,fulfillment_type,4,dropship,98.973020
25,vietnamese_tiki_products_fashion_accessories,fulfillment_type,3,dropship,98.058555
101,vietnamese_tiki_products_women_shoes,fulfillment_type,3,dropship,98.040209
82,vietnamese_tiki_products_women_bags,fulfillment_type,4,dropship,97.872832


### 2.6 `date_created` Inspection

In [89]:
# Summarize the distribution of date_created in each dataset

date_created_summary = []

for name, df in datasets.items():
    date_created_summary.append({
        "dataset": name,
        "min": df["date_created"].min(),
        "median": df["date_created"].median(),
        "max": df["date_created"].max(),
        "zero_values": (df["date_created"] == 0).sum(),
        "unique_values": df["date_created"].nunique()
    })

date_created_summary = pd.DataFrame(date_created_summary)

date_created_summary

,dataset,min,median,max,zero_values,unique_values
0,vietnamese_tiki_products_backpacks_suitcases,0,545.0,738076,20,1192
1,vietnamese_tiki_products_fashion_accessories,0,644.0,2260,32,1486
2,vietnamese_tiki_products_men_bags,0,642.0,2359,8,1079
3,vietnamese_tiki_products_men_shoes,0,608.0,2140,13,1053
4,vietnamese_tiki_products_women_bags,0,625.0,1984,7,1001
5,vietnamese_tiki_products_women_shoes,0,566.0,2889,11,1162


`date_created` is stored as an integer rather than a standard date format. Its exact meaning and unit are unclear, and some values are unusually large or equal to zero. Therefore, it is retained unchanged and is not interpreted as a calendar date in Week 1.

### Research Question 7 — Should unusual values or low-variation variables be removed immediately?

Unusual values should not be removed automatically. They should first be checked to determine whether they are real errors or valid observations.

`Unnamed: 0` is only a saved row index, so it can be removed from the processed dataset.

`favourite_count` is constant across the data and provides little analytical value. `fulfillment_type` is highly dominated by one value, but it is retained for exploratory comparison.

`date_created` has an unclear meaning and should be kept unchanged rather than interpreted as a datetime variable.

### 2.7 Data Quality Decisions

In [90]:
# Summarize data-quality findings and handling decisions

quality_decisions = pd.DataFrame([
    {
        "issue": "Unnamed: 0",
        "finding": "Matches the saved row index",
        "decision": "Remove from processed data"
    },
    {
        "issue": "Missing values",
        "finding": "3 missing cells",
        "decision": "Retain for now"
    },
    {
        "issue": "Duplicate product IDs",
        "finding": "Some IDs are duplicated",
        "decision": "Do not remove by ID alone"
    },
    {
        "issue": "Exact duplicate rows",
        "finding": "22 redundant rows after ignoring Unnamed: 0",
        "decision": "Remove redundant copies in processed data"
    },
    {
        "issue": "Zero price",
        "finding": "1 gift item in Women Bags",
        "decision": "Retain"
    },
    {
        "issue": "High prices",
        "finding": "Mostly plausible premium products",
        "decision": "Retain"
    },
    {
        "issue": "favourite_count",
        "finding": "Constant across records",
        "decision": "Exclude from analysis"
    },
    {
        "issue": "fulfillment_type",
        "finding": "Highly dominated by one category",
        "decision": "Retain for reference"
    },
    {
        "issue": "date_created",
        "finding": "Meaning and unit unclear",
        "decision": "Retain unchanged"
    },
    {
        "issue": "Category anomalies",
        "finding": "Some products appear in unexpected source groups",
        "decision": "Flag but retain"
    }
])

quality_decisions

,issue,finding,decision
0,Unnamed: 0,Matches the saved row index,Remove from processed data
1,Missing values,3 missing cells,Retain for now
2,Duplicate product IDs,Some IDs are duplicated,Do not remove by ID alone
3,Exact duplicate rows,22 redundant rows after ignoring Unnamed: 0,Remove redundant copies in processed data
4,Zero price,1 gift item in Women Bags,Retain
5,High prices,Mostly plausible premium products,Retain
6,favourite_count,Constant across records,Exclude from analysis
7,fulfillment_type,Highly dominated by one category,Retain for reference
8,date_created,Meaning and unit unclear,Retain unchanged
9,Category anomalies,Some products appear in unexpected source groups,Flag but retain


## 3. Data Preparation

In [91]:
# Create copies of the raw datasets for data preparation
processed_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

In [92]:
# Define readable source-group names for the six datasets
source_group_map = {
    "vietnamese_tiki_products_backpacks_suitcases": "Backpacks & Suitcases",
    "vietnamese_tiki_products_fashion_accessories": "Fashion Accessories",
    "vietnamese_tiki_products_men_bags": "Men Bags",
    "vietnamese_tiki_products_men_shoes": "Men Shoes",
    "vietnamese_tiki_products_women_bags": "Women Bags",
    "vietnamese_tiki_products_women_shoes": "Women Shoes"
}

In [93]:
# Remove the redundant index column and add the original product group
for name, df in processed_datasets.items():
    df.drop(columns=["Unnamed: 0"], inplace=True)
    df["source_group"] = source_group_map[name]

In [94]:
# Check the processed dataset structure and source-group labels
processed_summary = []

for name, df in processed_datasets.items():
    processed_summary.append({
        "source_group": source_group_map[name],
        "rows": df.shape[0],
        "columns": df.shape[1],
        "unnamed_removed": "Unnamed: 0" not in df.columns,
        "group_label_correct": df["source_group"].eq(source_group_map[name]).all()
    })

processed_summary = pd.DataFrame(processed_summary)

processed_summary

,source_group,rows,columns,unnamed_removed,group_label_correct
0,Backpacks & Suitcases,5361,19,True,True
1,Fashion Accessories,16019,19,True,True
2,Men Bags,4234,19,True,True
3,Men Shoes,5745,19,True,True
4,Women Bags,4325,19,True,True
5,Women Shoes,5919,19,True,True


### 3.1 Remove Exact Duplicate Rows

In [95]:
# Count exact duplicate rows in each processed dataset
duplicate_before = []

for name, df in processed_datasets.items():
    duplicate_before.append({
        "source_group": source_group_map[name],
        "rows": len(df),
        "redundant_duplicates": df.duplicated().sum()
    })

duplicate_before = pd.DataFrame(duplicate_before)

duplicate_before

,source_group,rows,redundant_duplicates
0,Backpacks & Suitcases,5361,2
1,Fashion Accessories,16019,10
2,Men Bags,4234,6
3,Men Shoes,5745,1
4,Women Bags,4325,1
5,Women Shoes,5919,2


In [96]:
# Remove exact duplicate rows while keeping the first occurrence
for name, df in processed_datasets.items():
    processed_datasets[name] = (
        df.drop_duplicates()
          .reset_index(drop=True)
    )

In [97]:
# Verify row counts and remaining exact duplicates after cleaning
duplicate_after = []

for name, df in processed_datasets.items():
    duplicate_after.append({
        "source_group": source_group_map[name],
        "rows_after": len(df),
        "remaining_duplicates": df.duplicated().sum()
    })

duplicate_after = pd.DataFrame(duplicate_after)

duplicate_after

,source_group,rows_after,remaining_duplicates
0,Backpacks & Suitcases,5359,0
1,Fashion Accessories,16009,0
2,Men Bags,4228,0
3,Men Shoes,5744,0
4,Women Bags,4324,0
5,Women Shoes,5917,0


### 3.2 Combine the Six Product Groups

In [98]:
# Combine all processed datasets into one DataFrame
combined_data = pd.concat(
    processed_datasets.values(),
    ignore_index=True
)

combined_data

,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold,source_group
0,179731375,Balo nữ da dầy thời trang unisex màu sắc tươi ...,Balo nữ da dầy thời trang unisex màu sắc tươ...,14250,14250,dropship,OEM,0,0.0,0,False,Thiên Ân Balo,488,10,0,False,Balo nữ,0,Backpacks & Suitcases
1,249221065,Ví da đựng thẻ ngân hàng (7 màu) - 9205,...,10260,10260,dropship,OEM,0,0.0,0,False,SHOP THỜI TRANG 2N,166,2,0,False,Root,0,Backpacks & Suitcases
2,102862196,Balo nam nữ thời trang họa tiết bò sữa kiểu m...,Balo họa tiết bò sữa thời trang cho nam...,19999,19999,dropship,OEM,0,0.0,0,False,Xưởng may balo Đà Nẵng,831,5,0,False,Root,0,Backpacks & Suitcases
3,115740385,Vải nhũ bóng simili chống thấm may túi ví balo...,_ Vải nhũ bóng siêu xinh và cực kì xịn ...,30000,30000,dropship,OEM,0,0.0,0,False,BB Outlet,775,4,0,False,Root,0,Backpacks & Suitcases
4,102862375,Combo balo vừa a4 và túi tote thời trang kiểu...,BALO THỜI TRANG - BALO ĐẸP – BALO HỌC S...,29000,29000,dropship,OEM,0,0.0,0,False,Xưởng may balo Đà Nẵng,831,5,0,False,Root,0,Backpacks & Suitcases
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41576,165961767,Xịt khử mùi giày dép chống hôi chân,- Đánh bay nỗi lo lắng phiền muộn về mùi hôi c...,55000,55000,dropship,\tOEM,0,0.0,0,False,A Bảo Store 88,559,6,0,False,Vệ sinh giày,0,Women Shoes
41577,253164871,Bộ kit CAO CẤP vệ sinh giày Sneaker Cleaning 1...,Bộ kit CAO CẤP vệ sinh giày Sneaker Cle...,129000,129000,dropship,OEM,0,0.0,0,True,Hàng Gia Dụng Q9,132,9,0,False,Root,0,Women Shoes
41578,202330632,Xi đánh giày Kangaroo Hàn Quốc-KGR1,Làm sạch và bảo vệ bề mặt da Xi đánh ...,18000,18000,dropship,OEM,0,0.0,0,False,happyjew,324,6,0,False,Root,0,Women Shoes
41579,252614724,"Gói 30 giấy lau giày, đồ đa Sneaker","Gói 30 khăn lau giày thể thao, giày da,...",39900,39900,dropship,OEM,0,0.0,0,False,Kho hàng Hạnh Phạm SG,143,9,1061,False,Root,0,Women Shoes


In [99]:
# Check the size of the combined dataset
print("Rows:", combined_data.shape[0])
print("Columns:", combined_data.shape[1])

Rows: 41581
Columns: 19


In [100]:
# Check the number of records in each source group
combined_data["source_group"].value_counts()

source_group
Fashion Accessories      16009
Women Shoes               5917
Men Shoes                 5744
Backpacks & Suitcases     5359
Women Bags                4324
Men Bags                  4228
Name: count, dtype: int64

In [101]:
# Preview the structure of the combined dataset
combined_data.head()

,id,name,description,original_price,price,fulfillment_type,brand,review_count,rating_average,favourite_count,pay_later,current_seller,date_created,number_of_images,vnd_cashback,has_video,category,quantity_sold,source_group
0,179731375,Balo nữ da dầy thời trang unisex màu sắc tươi ...,Balo nữ da dầy thời trang unisex màu sắc tươ...,14250,14250,dropship,OEM,0,0.0,0,False,Thiên Ân Balo,488,10,0,False,Balo nữ,0,Backpacks & Suitcases
1,249221065,Ví da đựng thẻ ngân hàng (7 màu) - 9205,...,10260,10260,dropship,OEM,0,0.0,0,False,SHOP THỜI TRANG 2N,166,2,0,False,Root,0,Backpacks & Suitcases
2,102862196,Balo nam nữ thời trang họa tiết bò sữa kiểu m...,Balo họa tiết bò sữa thời trang cho nam...,19999,19999,dropship,OEM,0,0.0,0,False,Xưởng may balo Đà Nẵng,831,5,0,False,Root,0,Backpacks & Suitcases
3,115740385,Vải nhũ bóng simili chống thấm may túi ví balo...,_ Vải nhũ bóng siêu xinh và cực kì xịn ...,30000,30000,dropship,OEM,0,0.0,0,False,BB Outlet,775,4,0,False,Root,0,Backpacks & Suitcases
4,102862375,Combo balo vừa a4 và túi tote thời trang kiểu...,BALO THỜI TRANG - BALO ĐẸP – BALO HỌC S...,29000,29000,dropship,OEM,0,0.0,0,False,Xưởng may balo Đà Nẵng,831,5,0,False,Root,0,Backpacks & Suitcases


### 3.3 Create Discount Variables

In [102]:
# Calculate the discount amount for each product
combined_data["discount_amount"] = (
    combined_data["original_price"] - combined_data["price"]
)

In [103]:
# Calculate discount percentage only when original_price is greater than zero
combined_data["discount_percentage"] = np.where(
    combined_data["original_price"] > 0,
    combined_data["discount_amount"]
    / combined_data["original_price"] * 100,
    np.nan
)

In [104]:
# Identify whether each product is currently discounted
combined_data["is_discounted"] = (
    combined_data["price"] < combined_data["original_price"]
)

In [105]:
# Preview the newly created discount variables
combined_data[
    [
        "original_price",
        "price",
        "discount_amount",
        "discount_percentage",
        "is_discounted"
    ]
].head(10)

,original_price,price,discount_amount,discount_percentage,is_discounted
0,14250,14250,0,0.0,False
1,10260,10260,0,0.0,False
2,19999,19999,0,0.0,False
3,30000,30000,0,0.0,False
4,29000,29000,0,0.0,False
5,22000,22000,0,0.0,False
6,28900,28900,0,0.0,False
7,25000,25000,0,0.0,False
8,29800,29800,0,0.0,False
9,49000,22050,26950,55.0,True


In [106]:
# Check the ranges and missing values of the derived discount variables
discount_check = pd.DataFrame({
    "min": combined_data[
        ["discount_amount", "discount_percentage"]
    ].min(),
    "max": combined_data[
        ["discount_amount", "discount_percentage"]
    ].max(),
    "missing": combined_data[
        ["discount_amount", "discount_percentage"]
    ].isna().sum()
})

discount_check

,min,max,missing
discount_amount,0.0,7.550000e+06,0
discount_percentage,0.0,8.857143e+01,1


In [108]:
# Count discounted and non-discounted products
combined_data["is_discounted"].value_counts()

is_discounted
False    32503
True      9078
Name: count, dtype: int64

### 3.4 Final Validation and Export

In [110]:
# Check the final size of the processed dataset
combined_data.shape

(41581, 22)

In [111]:
# Check that the redundant index column is no longer present
"Unnamed: 0" in combined_data.columns

False

In [112]:
# Check the six source groups in the combined dataset
combined_data["source_group"].value_counts()

source_group
Fashion Accessories      16009
Women Shoes               5917
Men Shoes                 5744
Backpacks & Suitcases     5359
Women Bags                4324
Men Bags                  4228
Name: count, dtype: int64

In [113]:
# Check the derived discount variables
combined_data[
    [
        "discount_amount",
        "discount_percentage",
        "is_discounted"
    ]
].describe(include="all")

,discount_amount,discount_percentage,is_discounted
count,4.158100e+04,41580.000000,41581
unique,NaN,NaN,2
top,NaN,NaN,False
freq,NaN,NaN,32503
mean,3.261636e+04,5.805697,NaN
std,2.024304e+05,13.717197,NaN
min,0.000000e+00,0.000000,NaN
25%,0.000000e+00,0.000000,NaN
50%,0.000000e+00,0.000000,NaN
75%,0.000000e+00,0.000000,NaN


In [114]:
# Summarize the final validation checks before export
final_validation = pd.DataFrame([
    {
        "check": "Total rows",
        "result": len(combined_data)
    },
    {
        "check": "Total columns",
        "result": combined_data.shape[1]
    },
    {
        "check": "Exact duplicates",
        "result": combined_data.duplicated().sum()
    },
    {
        "check": "Source groups",
        "result": combined_data["source_group"].nunique()
    },
    {
        "check": "Unnamed: 0 present",
        "result": "Unnamed: 0" in combined_data.columns
    }
])

final_validation

,check,result
0,Total rows,41581
1,Total columns,22
2,Exact duplicates,0
3,Source groups,6
4,Unnamed: 0 present,False


In [115]:
# Save the cleaned and combined dataset to the processed-data folder
processed_file = PROCESSED_DATA_PATH / "tiki_products_combined.csv"

combined_data.to_csv(
    processed_file,
    index=False
)

In [116]:
# Confirm that the processed CSV file was created successfully
print("Saved file:", processed_file)
print("File exists:", processed_file.exists())

Saved file: ..\data\processed\tiki_products_combined.csv
File exists: True
